In [2]:
import json
from pathlib import Path

import pandas as pd

# ============================================================
# CONFIG
# ============================================================

INPUT_CSV = Path("./../../MIMICEL_data/mimicel_train.csv")
ACTIVITY2ID_JSON = Path("./activity2id.json")

CASE_COL = "stay_id"
ACT_COL = "activity"
TIME_COL = "timestamps"

OUT_DIR = Path("./results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(INPUT_CSV)

with open(ACTIVITY2ID_JSON, "r", encoding="utf-8") as f:
    activity2id = json.load(f)

# JSON 값이 문자열일 가능성 방지
activity2id = {
    str(act): int(idx)
    for act, idx in activity2id.items()
}

# timestamp 정렬
if TIME_COL in df.columns:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.sort_values([CASE_COL, TIME_COL]).reset_index(drop=True)
else:
    df = df.sort_values([CASE_COL]).reset_index(drop=True)

# activity id 매핑
df["activity_id"] = df[ACT_COL].map(activity2id)

# 매핑 실패 확인
missing = df[df["activity_id"].isna()][ACT_COL].unique()
if len(missing) > 0:
    raise ValueError(f"activity2id.json에 없는 activity가 있습니다: {missing}")

df["activity_id"] = df["activity_id"].astype(int)

# ============================================================
# 1. act_dict.csv
# format: activity,index
# ============================================================

act_dict_df = pd.DataFrame([
    [act, idx]
    for act, idx in sorted(activity2id.items(), key=lambda x: x[1])
])

act_dict_path = OUT_DIR / "act_dict.csv"
act_dict_df.to_csv(
    act_dict_path,
    header=False,
    index=False,
    encoding="utf-8"
)

# ============================================================
# 2. act_freq_dict.csv
# format: activity_id,frequency
# ============================================================

act_freq = (
    df["activity_id"]
    .value_counts()
    .sort_index()
)

act_freq_df = pd.DataFrame([
    [int(activity_id), int(freq)]
    for activity_id, freq in act_freq.items()
])

act_freq_path = OUT_DIR / "act_freq_dict.csv"
act_freq_df.to_csv(
    act_freq_path,
    header=False,
    index=False,
    encoding="utf-8"
)

# ============================================================
# 3. length_dict.csv
# format: trace_length,count
# ============================================================

trace_lengths = (
    df.groupby(CASE_COL)
      .size()
)

length_dist = (
    trace_lengths
    .value_counts()
    .sort_index()
)

length_df = pd.DataFrame([
    [int(trace_len), int(count)]
    for trace_len, count in length_dist.items()
])

length_path = OUT_DIR / "length_dict.csv"
length_df.to_csv(
    length_path,
    header=False,
    index=False,
    encoding="utf-8"
)

# ============================================================
# VALIDATION
# ============================================================

print("===================================")
print("ProcessGAN data_info files created")
print("===================================")
print("Input CSV:", INPUT_CSV)
print("Cases:", df[CASE_COL].nunique())
print("Events:", len(df))
print("Activities:", df["activity_id"].nunique())
print("Min activity id:", df["activity_id"].min())
print("Max activity id:", df["activity_id"].max())
print("Max trace length:", trace_lengths.max())
print("===================================")

print("\nSaved files:")
print(act_dict_path)
print(act_freq_path)
print(length_path)

print("\nact_dict.csv preview")
print(act_dict_df)

print("\nact_freq_dict.csv preview")
print(act_freq_df.head())

print("\nlength_dict.csv preview")
print(length_df.head())

ProcessGAN data_info files created
Input CSV: ..\..\MIMICEL_data\mimicel_train.csv
Cases: 899
Events: 16176
Activities: 6
Min activity id: 1
Max activity id: 6
Max trace length: 100

Saved files:
results\act_dict.csv
results\act_freq_dict.csv
results\length_dict.csv

act_dict.csv preview
                         0  1
0    Discharge from the ED  1
1             Enter the ED  2
2   Medicine dispensations  3
3  Medicine reconciliation  4
4         Triage in the ED  5
5         Vital sign check  6

act_freq_dict.csv preview
   0     1
0  1  1916
1  2   899
2  3  3085
3  4  6345
4  5   899

length_dict.csv preview
   0   1
0  3   6
1  4  21
2  5  36
3  6  48
4  7  40
